In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
data_train = pd.read_csv("/Users/nethanshaik/Desktop/kaggle_competiion/data/train.csv")
data_test = pd.read_csv("/Users/nethanshaik/Desktop/kaggle_competiion/data/test.csv")

In [ ]:
data_train.head()

In [ ]:
data_test.head()

In [ ]:
data_train.info()

In [ ]:
data_test.info()

In [ ]:
data_train = data_train.drop(['id'], axis=1)

In [ ]:
data_train.head()

In [ ]:
data_test.head()

In [ ]:
data_train.isnull().any()

In [ ]:
data_test.isnull().sum().sum()

In [ ]:
cols = ["Soil_Type","Crop_Type","Crop_Growth_Stage","Season", "Irrigation_Type", "Water_Source", "Mulching_Used", "Region", "Irrigation_Need"]
unique_value_summary = data_train[cols].apply(lambda x: x.unique())
print(unique_value_summary)

In [ ]:
X_train = data_train.drop(['Irrigation_Need'], axis=1)
X_test = data_test
y_train = data_train['Irrigation_Need']

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
categorical_cols = ["Soil_Type","Crop_Type","Season","Irrigation_Type","Water_Source","Region"]
ordinal_cols = X_train['Crop_Growth_Stage'].unique()
print(ordinal_cols)

In [ ]:
X_train = pd.get_dummies(X_train, columns=categorical_cols, dtype=int)
X_test = pd.get_dummies(X_test, columns=categorical_cols, dtype=int)

orec = OrdinalEncoder(categories=[ordinal_cols])
X_train["Crop_Growth_Stage"] = orec.fit_transform(X_train[["Crop_Growth_Stage"]])
X_test["Crop_Growth_Stage"] = orec.transform(X_test[["Crop_Growth_Stage"]])

X_train["Mulching_Used"] = X_train["Mulching_Used"].map({"Yes": 1, "No": 0})
X_test["Mulching_Used"] = X_test["Mulching_Used"].map({"Yes": 1, "No": 0})

y_train = y_train.map({"Low":0, "Medium":1, "High":2})

In [ ]:
pd.set_option('display.max_columns', None)
X_train.head()

In [ ]:
y_train.head()

In [ ]:
X_test.head()

In [ ]:
X_train.columns

In [ ]:
models = {
    'Random Forest':(RandomForestClassifier(random_state=42), {
        'n_estimators':[100,200, 300],
        'criterion':['gini','entropy'],
        'max_depth': [2,3, 5],
        'min_samples_split': [2,3],
        'min_samples_leaf': [1,2],
    }),

    'XGBoost':(XGBClassifier(random_state=42, eval_metric='mlogloss'), {
        'n_estimators':[100,200, 300],
        'learning_rate':[0.01, 0.05,0.1],
        'max_depth':[2,3,5],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
    }),

}

In [ ]:
best_models = {}
results = []

for name, (model, params) in models.items():
    print(f"\n Running {name}")
    grid = GridSearchCV(
        estimator=model,
        param_grid=params,
        scoring='accuracy',
        cv=5,
        n_jobs=-1,
        return_train_score=True,
        verbose=True,
    )

    grid.fit(X_train, y_train)

In [ ]:
test_ids = X_test["id"]
X_test = X_test.drop("id", axis=1)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print('Best Parameters :', grid.best_params_)
print('CV best score: ', grid.best_score_)


In [ ]:
reverse_map = {0: "Low", 1: "Medium", 2: "High"}

submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": pd.Series(y_pred).map(reverse_map)
})

In [ ]:
submission.to_csv("submission.csv", index=False)